# Interactive per-PDB parameters editor

Use the widget below to view, add, edit, and save per-PDB CPOINT/CVECT values to pdb_params.yaml. Requires pyyaml and ipywidgets.

In [ ]:
import pathlib
import yaml
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

p = pathlib.Path('pdb_params.yaml')
cfg = yaml.safe_load(p.read_text()) if p.exists() else {}

def make_df(data):
    rows = []
    for name, vals in (data or {}).items():
        rows.append({
            'pdb': name,
            'CPOINT': vals.get('CPOINT'),
            'CVECT': vals.get('CVECT'),
        })
    return pd.DataFrame(rows)

out = widgets.Output()
df_widget = widgets.Output()

def refresh_display():
    with df_widget:
        clear_output()
        display(make_df(cfg))

refresh_display()
display(df_widget)


In [ ]:
# Interactive editor widgets
pdb_names = sorted(cfg.keys())
sel = widgets.Dropdown(options=pdb_names, description='PDB:')
cpoint_in = widgets.Text(description='CPOINT (csv)')
cvect_in = widgets.Text(description='CVECT (csv)')
status = widgets.HTML('')

def parse_csv_list(s):
    if s is None:
        return None
    s = str(s).strip()
    if s == '':
        return None
    parts = [p for p in s.replace(',', ' ').split() if p]
    try:
        return [float(x) for x in parts]
    except ValueError:
        return parts

def on_select(change=None):
    name = sel.value
    if not name:
        cpoint_in.value = ''
        cvect_in.value = ''
        return
    vals = cfg.get(name, {})
    cp = vals.get('CPOINT')
    cv = vals.get('CVECT')
    cpoint_in.value = ', '.join(map(str, cp)) if cp else ''
    cvect_in.value = ', '.join(map(str, cv)) if cv else ''
    status.value = ''

sel.observe(on_select, names='value')

def save_entry(b):
    name = sel.value
    if not name:
        status.value = '<b style="color:red">No PDB selected</b>'
        return
    cp = parse_csv_list(cpoint_in.value)
    cv = parse_csv_list(cvect_in.value)
    entry = {}
    if cp is not None:
        entry['CPOINT'] = cp
    if cv is not None:
        entry['CVECT'] = cv
    cfg[name] = entry
    p.write_text(yaml.safe_dump(cfg))
    status.value = '<b style="color:green">Saved</b>'
    refresh_display()

def add_entry(b):
    new_name = new_name_in.value.strip()
    if not new_name:
        status.value = '<b style="color:red">Provide name</b>'
        return
    if new_name in cfg:
        status.value = '<b style="color:orange">Already exists</b>'
        return
    cfg[new_name] = {}
    p.write_text(yaml.safe_dump(cfg))
    sel.options = sorted(cfg.keys())
    sel.value = new_name
    status.value = '<b style="color:green">Added</b>'
    refresh_display()

def delete_entry(b):
    name = sel.value
    if not name or name not in cfg:
        status.value = '<b style="color:red">Nothing to delete</b>'
        return
    del cfg[name]
    p.write_text(yaml.safe_dump(cfg))
    sel.options = sorted(cfg.keys())
    sel.value = None
    status.value = '<b style="color:green">Deleted</b>'
    refresh_display()

save_btn = widgets.Button(description='Save', button_style='success')
add_btn = widgets.Button(description='Add', button_style='info')
del_btn = widgets.Button(description='Delete', button_style='danger')
new_name_in = widgets.Text(description='New name')

save_btn.on_click(save_entry)
add_btn.on_click(add_entry)
del_btn.on_click(delete_entry)

ui = widgets.VBox([widgets.HBox([sel, new_name_in, add_btn, del_btn]), cpoint_in, cvect_in, widgets.HBox([save_btn, status])])
display(ui)
on_select()
